In [2]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Display plots inside the notebook
%matplotlib inline

In [3]:
# Load the dataset
df = pd.read_csv("finance_dataset.csv")

print("Dataset loaded successfully.")
print("Shape:", df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'finance_dataset.csv'

In [ ]:
# Dataset structure
print("Shape of dataset:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
# Basic statistical analysis
df.describe()

In [ ]:
# Target class distribution
print("Loan default counts:")
print(df["loan_default"].value_counts())

print("\nLoan default percentages:")
print((df["loan_default"].value_counts(normalize=True) * 100).round(2))

In [ ]:
# Age vs Loan Default
plt.figure(figsize=(8, 5))
sns.boxplot(x="loan_default", y="age", data=df)
plt.title("Age vs Loan Default")
plt.xlabel("Loan Default (0 = No, 1 = Yes)")
plt.ylabel("Age")
plt.show()

In [ ]:
# Credit Score Distribution
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="credit_score", hue="loan_default", kde=True, bins=30)
plt.title("Credit Score Distribution by Loan Default")
plt.xlabel("Credit Score")
plt.ylabel("Number of Customers")
plt.show()

In [ ]:
# Annual Income Distribution
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="annual_income", hue="loan_default", kde=True, bins=30)
plt.title("Annual Income Distribution by Loan Default")
plt.xlabel("Annual Income")
plt.ylabel("Number of Customers")
plt.show()

In [ ]:
# Loan Amount vs Loan Default
plt.figure(figsize=(8, 5))
sns.boxplot(x="loan_default", y="loan_amount", data=df)
plt.title("Loan Amount vs Loan Default")
plt.xlabel("Loan Default (0 = No, 1 = Yes)")
plt.ylabel("Loan Amount")
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(9, 6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# Separate features (X) and target (y)
# customer_id is excluded because it is only a unique identifier.

features = [
    "age",
    "annual_income",
    "credit_score",
    "loan_amount",
    "existing_loans"
]

X = df[features]
y = df["loan_default"]

print("Features:")
print(X.columns.tolist())
print("\nTarget: loan_default")

In [ ]:
# Train-test split: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

In [ ]:
# Feature scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed.")

In [ ]:
# Train Logistic Regression classifier
model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

model.fit(X_train_scaled, y_train)

print("Logistic Regression model trained successfully.")

In [ ]:
# Generate predictions
y_pred = model.predict(X_test_scaled)

# Default probability for each test record
y_probability = model.predict_proba(X_test_scaled)[:, 1]

print("First 20 predictions:")
print(y_pred[:20])

print("\nFirst 20 default probabilities:")
print(np.round(y_probability[:20], 4))

In [ ]:
# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("========== MODEL PERFORMANCE ==========")
print(f"Accuracy : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall   : {recall:.4f} ({recall*100:.2f}%)")
print(f"F1 Score : {f1:.4f} ({f1*100:.2f}%)")

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["No Default", "Default"],
    yticklabels=["No Default", "Default"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Full classification report
print(classification_report(
    y_test,
    y_pred,
    target_names=["No Default", "Default"],
    zero_division=0
))

In [ ]:
# Examine feature coefficients
coefficients = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.coef_[0]
}).sort_values(by="Coefficient", ascending=False)

coefficients